In [7]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import optuna
import lightgbm as lgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)
os.makedirs("../results", exist_ok=True)
os.makedirs("../figures", exist_ok=True)
print("Environment initialized with SEED =", SEED)

Environment initialized with SEED = 42


In [8]:
df = pd.read_csv("../data/Churn_Modelling.csv")

X = df.drop(columns=["RowNumber", "CustomerId", "Surname", "Exited"])
y = df["Exited"].values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)

cat_cols = ["Geography", "Gender"]
num_cols = [c for c in X.columns if c not in cat_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first", sparse_output=False), cat_cols),
        ("num", "passthrough", num_cols),
    ]
)

X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

print(f"Train shape: {X_train.shape} | Test shape: {X_test.shape}")

Train shape: (8000, 11) | Test shape: (2000, 11)


In [11]:
def measure_latency_ms(model, X_sample, n_repeats=15):
    for _ in range(3):
        _ = model.predict_proba(X_sample)
    latencies = []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        _ = model.predict_proba(X_sample)
        t1 = time.perf_counter()
        latencies.append((t1 - t0) * 1000.0)
    return np.median(latencies) * (1000.0 / len(X_sample))

def evaluate_config(params):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    aucs, latencies = [], []
    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr, y_tr = X_train[train_idx], y_train[train_idx]
        X_va, y_va = X_train[val_idx], y_train[val_idx]
        
        clf = lgb.LGBMClassifier(**params, random_state=SEED, n_jobs=-1, verbose=-1)
        clf.fit(X_tr, y_tr)
        
        preds = clf.predict_proba(X_va)[:, 1]
        aucs.append(roc_auc_score(y_va, preds))
        latencies.append(measure_latency_ms(clf, X_va, n_repeats=15))
    return float(np.mean(aucs)), float(np.mean(latencies))

# 1. Baseline 1: Default LightGBM
default_params = {
    "n_estimators": 100,
    "max_depth": -1,
    "learning_rate": 0.1,
    "num_leaves": 31,
    "min_child_samples": 20,
}
def_auc, def_lat = evaluate_config(default_params)
pd.DataFrame([{
    "method": "Default_LightGBM",
    "trial": 0,
    **default_params,
    "cv_roc_auc": def_auc,
    "cv_latency_ms_per_1k": def_lat
}]).to_csv("../results/baseline_default.csv", index=False)

print(f"[Baseline 1 - Default] CV ROC-AUC: {def_auc:.4f} | Latency: {def_lat:.4f} ms/1k")

# 2. Baseline 2: Random Search (40 trials - có thanh tiến trình)
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 127),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
    }
    return evaluate_config(params)

study_random = optuna.create_study(
    directions=["maximize", "minimize"],
    sampler=optuna.samplers.RandomSampler(seed=SEED)
)
study_random.optimize(objective, n_trials=40, show_progress_bar=True)

df_random = study_random.trials_dataframe()
df_random["method"] = "Random_Search"
df_random.to_csv("../results/baseline_random_search.csv", index=False)
print("[Baseline 2 - Random Search] Completed 40 trials.")

[Baseline 1 - Default] CV ROC-AUC: 0.8606 | Latency: 3.7563 ms/1k


  0%|          | 0/40 [00:00<?, ?it/s]

[Baseline 2 - Random Search] Completed 40 trials.


In [12]:
# 3. Proposed Method: Optuna NSGA-II (40 trials)
study_nsga2 = optuna.create_study(
    directions=["maximize", "minimize"],
    sampler=optuna.samplers.NSGAIISampler(population_size=10, seed=SEED)
)
study_nsga2.optimize(objective, n_trials=40, show_progress_bar=True)

df_nsga2 = study_nsga2.trials_dataframe()
df_nsga2["method"] = "NSGA_II"
df_nsga2.to_csv("../results/nsga2_trials.csv", index=False)

# Trích xuất tập nghiệm không bị trội (Pareto-optimal front)
pareto_trials = study_nsga2.best_trials
pareto_records = []
for t in pareto_trials:
    pareto_records.append({
        "trial": t.number,
        **t.params,
        "cv_roc_auc": t.values[0],
        "cv_latency_ms_per_1k": t.values[1]
    })

df_pareto = pd.DataFrame(pareto_records).sort_values(by="cv_roc_auc", ascending=False)
df_pareto.to_csv("../results/nsga2_pareto_front.csv", index=False)

print(f"[Proposed - NSGA-II] Found {len(df_pareto)} Pareto-optimal solutions:")
display(df_pareto)

  0%|          | 0/40 [00:00<?, ?it/s]

[Proposed - NSGA-II] Found 5 Pareto-optimal solutions:


,trial,n_estimators,max_depth,learning_rate,num_leaves,min_child_samples,cv_roc_auc,cv_latency_ms_per_1k
3,25,414,4,0.027011,92,35,0.864644,11.734150
4,30,83,3,0.286914,82,42,0.864008,2.618512
0,1,120,3,0.190304,82,74,0.861996,2.580612
1,2,59,12,0.169675,38,26,0.854894,2.520112
2,21,70,4,0.030238,58,74,0.854541,2.186150
